# 05 Compare oatmilk trial results to Pioneer-21 cast statistics

## Import modules

In [ ]:
from os import path
import glob
import re
import ast
import cmocean.cm as cmo
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from sklearn import preprocessing

## Load local oatmilk trial data

In [ ]:
# Define paths to data and find files matching the file name pattern
LISST_PATH = "C:/Users/kylene.cooley/Documents/prtsz_bench_test"
DATA_PATH = "oatmilk-*[0-9]percent.csv"

flist = glob.glob(DATA_PATH, root_dir=LISST_PATH, recursive=True)

In [ ]:
# Define functions
def load_lisst(fpath, csvhdr):
    # Load processed LISST data
    lisst_data = pd.read_csv(fpath, names=csvhdr)
    try: lisst_data.head(1) 
    except: print("No LISST data found.")
    # Create LISST time vectors for Xarray Dataset coordinates
    lisst_time = pd.to_datetime(
         lisst_data[["year", "month", "day", "hour", "minute", "second"]],
         yearfirst=True, utc=True
         )
    lisst_data.insert(0, "time", lisst_time.values)
    lisst_data.set_index("time", drop=True, inplace=True)
    # print(lisst_data.head(1))
    # Convert data frames to xarray for easy manipulation
    lisst_ds = xr.Dataset.from_dataframe(lisst_data)
    # Create 2D array for binned volume concentration
    volumecon2D = list([])
    bins = list([])
    for var in lisst_ds.variables:
        if re.search("volumecon[0-9]+", var):
                bins.append(var)
                volumecon2D.append(lisst_ds[var])
    lisst_ds = lisst_ds.drop_vars(bins)
    str2num = lambda x: int(x.replace("volumecon", ""))
    bins = [str2num(x) for x in bins]
    lisst_ds["volume_concentration_2D"] = xr.concat(
        volumecon2D, pd.Index(bins, name="bin")
        )
    lisst_ds["volume_concentration_2D"] = lisst_ds["volume_concentration_2D"].assign_attrs(units="$\mu$L/L")
    return lisst_ds

In [ ]:
# Load LISST CSV column names from json containing column headers
headers = pd.read_json("./inst_headers/lisst_hdr.json", typ='series', orient='records')
csvhdr = headers.iloc[0]

In [ ]:
# Change first 36 column names to integer bin numbers
# bins = np.arange(36)+1
# csvhdr[0:36] = bins.astype(str)
# csvhdr

In [ ]:
# Load data to workspace
lisst0017 = load_lisst(path.join(LISST_PATH, flist[0]), csvhdr)
lisst0366 = load_lisst(path.join(LISST_PATH, flist[1]), csvhdr)
lisst0776 = load_lisst(path.join(LISST_PATH, flist[2]), csvhdr)
lisst1240 = load_lisst(path.join(LISST_PATH, flist[3]), csvhdr)
lisst1776 = load_lisst(path.join(LISST_PATH, flist[4]), csvhdr)
lisst2409 = load_lisst(path.join(LISST_PATH, flist[5]), csvhdr)
lisst3185 = load_lisst(path.join(LISST_PATH, flist[6]), csvhdr)
lisst4185 = load_lisst(path.join(LISST_PATH, flist[7]), csvhdr)

## Statistical summary of oatmilk results

In [ ]:
# Mean optical transmission
optical_transmission_avg = np.array([lisst0017.optical_transmission.mean(), lisst0366.optical_transmission.mean(),
       lisst0776.optical_transmission.mean(), lisst1240.optical_transmission.mean(),
       lisst1776.optical_transmission.mean(), lisst2409.optical_transmission.mean(),
       lisst3185.optical_transmission.mean(), lisst4185.optical_transmission.mean()])*100
print(optical_transmission_avg)

In [ ]:
# Mean total volume concentration
total_volume_avg = np.array([lisst0017.total_volumecon.mean(), lisst0366.total_volumecon.mean(),
       lisst0776.total_volumecon.mean(), lisst1240.total_volumecon.mean(),
       lisst1776.total_volumecon.mean(), lisst2409.total_volumecon.mean(),
       lisst3185.total_volumecon.mean(), lisst4185.total_volumecon.mean()])
print(total_volume_avg)

## Load statistics from LISST cast data

## Comparing bench test samples to Pioneer-21 cast data

In [ ]:
# Plot avg optical transmission against
# avg total volume concentration
fig3, ax3 = plt.subplots()
ax3.scatter(optical_transmission_avg, total_volume_avg)
ax3.set_xlabel("Optical Transmission %")
ax3.set_ylabel("Volume Concentration [$\mu$L/L]")
plt.title("Average total volume concentration and optical transmission\nLISST Bench Testing - July 2025")
ax3.grid()